In [1]:
from __future__ import annotations

import numpy as np
import sympy as sp
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from config.CEQLConfig import ModelTrainingConfig, CEQLConfig
from src.ComplexEQL import ComplexEQL
from src.utils import set_seed, train
from src.sympy_utils import filter_imaginary_part

# Dataset

In [2]:
def generate_xy_dataset(
    n_samples: int = 1000,
    x_range: tuple[float, float] = (-5.0, 5.0),   # x>0 to keep x^y real
    y_range: tuple[float, float] = (-5.0, 5.0),
):
    """
    Generate dataset for f(x, y) = x^y.
    We restrict x>0 so that x^y is real-valued for real y.
    Also filter out samples with |f| > 20 to avoid huge targets.
    """
    # sample x, y independently
    x = np.random.uniform(*x_range, size=(n_samples, 1))
    y = np.random.uniform(*y_range, size=(n_samples, 1))

    # compute function: f = x^y
    f = x / (x + y) # shape (N,1)

    # filter out non-finite values (just in case) and |f| > 20
    mask = np.isfinite(f) & (np.abs(f) <= 100.0)
    x = x[mask]
    y = y[mask]
    f = f[mask]

    # reshape again so concatenation never fails
    x = x.reshape(-1, 1)
    y = y.reshape(-1, 1)
    f = f.reshape(-1, 1)

    # concatenate safely: X = [x, y]
    X = np.concatenate([x, y], axis=1)

    return torch.from_numpy(X).float(), torch.from_numpy(f).float()

# Training

In [3]:
# -------------------------
# Config and setup
# -------------------------
mcfg = ModelTrainingConfig()
ncfg = CEQLConfig()

device = torch.device(mcfg.device)
set_seed(42)

# -------------------------
# Data
# -------------------------
X, y = generate_xy_dataset()
assert torch.isfinite(X).all(), "X contains NaN/Inf"
assert torch.isfinite(y).all(), "y contains NaN/Inf"
dataset = TensorDataset(X, y)
dataloader = DataLoader(
    dataset,
    batch_size=mcfg.train_batch_size,
    shuffle=True,
    drop_last=False,
)

# -------------------------
# Model, loss, optimizer, scheduler
# -------------------------
model = ComplexEQL(ncfg).to(device)
loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=mcfg.lr)

scheduler = None
if getattr(mcfg, "scheduler", None) == "ReduceLROnPlateau":
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, **mcfg.schedulerparams
    )
elif getattr(mcfg, "scheduler", None) is None:
    scheduler = None
else:
    raise ValueError(f"Unknown scheduler: {mcfg.scheduler}")

# -------------------------
# Training (cycle-based; scheduler is stepped only in final 10000 epochs inside train)
# -------------------------
model, (sl0_weights, al_weights, imaginary_out_losses, data_losses) = train(
    model=model,
    dataloader=dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    cfg=mcfg,
    device=device,
    scheduler=scheduler,
)

# -------------------------
# Symbolic readout
# -------------------------
x_sym, y_sym = sp.symbols("x y")
symbolic_expr = model.get_symbolic_expression([x_sym, y_sym], rounding_decimals=2)
print("\nDiscovered expression:")
print(symbolic_expr)

Random seed set as 42
[RAMP | Epoch 1 | cycle=0] lr=1.00e-03, total=4.1435e+01, data=4.1432e+01, sparsity_reg=0.0000e+00, imag_w=3.7166e-03, r=0.0100, active_edges=45
[RAMP | Epoch 100 | cycle=0] lr=1.00e-03, total=3.9261e+01, data=3.9259e+01, sparsity_reg=0.0000e+00, imag_w=2.3073e-03, r=0.0105, active_edges=45
[RAMP | Epoch 200 | cycle=0] lr=1.00e-03, total=3.8732e+01, data=3.8730e+01, sparsity_reg=0.0000e+00, imag_w=1.6252e-03, r=0.0110, active_edges=45
[RAMP | Epoch 300 | cycle=0] lr=1.00e-03, total=3.6883e+01, data=3.6881e+01, sparsity_reg=0.0000e+00, imag_w=1.1864e-03, r=0.0115, active_edges=45
[RAMP | Epoch 400 | cycle=0] lr=1.00e-03, total=3.6422e+01, data=3.6421e+01, sparsity_reg=0.0000e+00, imag_w=8.7920e-04, r=0.0120, active_edges=45
[RAMP | Epoch 500 | cycle=0] lr=1.00e-03, total=3.8024e+01, data=3.8024e+01, sparsity_reg=0.0000e+00, imag_w=9.1468e-04, r=0.0126, active_edges=45
[RAMP | Epoch 600 | cycle=0] lr=1.00e-03, total=3.5479e+01, data=3.5478e+01, sparsity_reg=0.0000e+

KeyboardInterrupt: 

In [ ]:
x_sym, y_sym = sp.symbols("x y")
symbolic_expr = model.get_symbolic_expression([x_sym, y_sym], rounding_decimals=2)
# symbolic_expr = filter_imaginary_part(symbolic_expr, [x_sym, y_sym])
print(symbolic_expr)

In [ ]:
symbolic_expr

In [ ]:
sp.simplify(symbolic_expr)

# Visualize the predictions

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # OK to keep

# -------------------------------------------------------
# Grid over (x, y) for visualization
#   Match your data generation ranges
# -------------------------------------------------------
x_min, x_max = -50.0, 50.0
y_min, y_max = -50.0, 50.0
grid_res = 200

xs = np.linspace(x_min, x_max, grid_res)
ys = np.linspace(y_min, y_max, grid_res)
Xg, Yg = np.meshgrid(xs, ys)

grid = np.stack([Xg, Yg], axis=-1)  # (R, R, 2)
grid_t = torch.from_numpy(grid.reshape(-1, 2)).float().to(device)

# -------------------------------------------------------
# IMPORTANT: r is REQUIRED by your current surrogate op
# Choose r=1.0 to visualize the final (true) sin behavior,
# or use your scheduled r to see the "early training" surrogate.
# -------------------------------------------------------
r_vis = 1.0  # final / exact sin mode

# -------------------------------------------------------
# Model predictions on grid
# -------------------------------------------------------
model.eval()
with torch.no_grad():
    pred = model(grid_t, r=r_vis)  # (R*R, 1), complex
    pred_real = pred.real.detach().cpu().numpy().reshape(grid_res, grid_res)

# -------------------------------------------------------
# Ground truth: f(x,y) = sin(x)  (your dataset)
# -------------------------------------------------------
f_true = np.sin(Xg)

# -------------------------------------------------------
# Shared limits (optional) to make surfaces comparable
# -------------------------------------------------------
z_min = np.min([pred_real.min(), f_true.min()])
z_max = np.max([pred_real.max(), f_true.max()])

# -------------------------------------------------------
# 3D plot: prediction surface
# -------------------------------------------------------
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(Xg, Yg, pred_real, cmap="viridis", linewidth=0, antialiased=True)

ax.set_title(f"Model Prediction Surface (real part), r={r_vis}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("f_pred")
ax.set_zlim(z_min, z_max)

plt.tight_layout()
plt.show()

# -------------------------------------------------------
# 3D plot: ground truth surface
# -------------------------------------------------------
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(Xg, Yg, f_true, cmap="plasma", linewidth=0, antialiased=True)

ax.set_title("Ground Truth Surface  f(x,y) = sin(x)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("f_true")
ax.set_zlim(z_min, z_max)

plt.tight_layout()
plt.show()

# -------------------------------------------------------
# 3D plot: prediction error surface
# -------------------------------------------------------
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

error = pred_real - f_true
ax.plot_surface(Xg, Yg, error, cmap="coolwarm", linewidth=0, antialiased=True)

ax.set_title("Prediction Error Surface  (model - truth)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("error")

plt.tight_layout()
plt.show()

# -------------------------------------------------------
# 2D diagnostic: multiple y-slices to verify y-invariance
# -------------------------------------------------------
x_line = np.linspace(x_min, x_max, 1200)
y_slices = [-40.0, 0.0, 40.0]

plt.figure(figsize=(10, 4))
for y0 in y_slices:
    X_line = np.stack([x_line, np.full_like(x_line, y0)], axis=1)
    X_line_t = torch.from_numpy(X_line).float().to(device)

    model.eval()
    with torch.no_grad():
        pred_line = model(X_line_t, r=r_vis).real.detach().cpu().numpy().reshape(-1)

    plt.plot(x_line, pred_line, label=f"pred (y={y0:g})")

plt.plot(x_line, np.sin(x_line), linestyle="--", label="true sin(x)")
plt.xlabel("x")
plt.ylabel("f(x, y0)")
plt.title(f"Y-invariance check (real part), r={r_vis}")
plt.legend()
plt.tight_layout()
plt.show()


# Visualize weights

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Helper: flatten list of complex numpy arrays
# ----------------------------
def flatten_complex_list(weight_list):
    flat = [w.reshape(-1) for w in weight_list]
    mat = np.vstack(flat)
    return mat.real, mat.imag


# ==========================================================
# 1) Symbolic layer 0 weights
# ==========================================================
sl0_real, sl0_imag = flatten_complex_list(sl0_weights)
epochs0 = np.arange(len(sl0_real))

in_dim0, out_dim0 = sl0_weights[0].shape
labels_sl0 = [f"{i}{j}" for i in range(in_dim0) for j in range(out_dim0)]

# --- real part ---
plt.figure(figsize=(8, 5))
for k in range(sl0_real.shape[1]):
    plt.plot(epochs0, sl0_real[:, k], label=labels_sl0[k])
plt.title("Symbolic Layer 0 Weights (Real)")
plt.xlabel("Epoch")
plt.ylabel("Real part")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize="small")
plt.tight_layout()

# --- imaginary part ---
plt.figure(figsize=(8, 5))
for k in range(sl0_imag.shape[1]):
    plt.plot(epochs0, sl0_imag[:, k], label=labels_sl0[k])
plt.title("Symbolic Layer 0 Weights (Imag)")
plt.xlabel("Epoch")
plt.ylabel("Imag part")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize="small")
plt.tight_layout()


# ==========================================================
# 2) Assembly layer weights
# ==========================================================
al_real, al_imag = flatten_complex_list(al_weights)
epochs_al = np.arange(len(al_real))

labels_al = [f"{i:02d}" for i in range(al_real.shape[1])]

# --- real part ---
plt.figure(figsize=(8, 5))
for k in range(al_real.shape[1]):
    plt.plot(epochs_al, al_real[:, k], label=labels_al[k])
plt.title("Assembly Layer Weights (Real)")
plt.xlabel("Epoch")
plt.ylabel("Real part")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize="small")
plt.tight_layout()

# --- imaginary part ---
plt.figure(figsize=(8, 5))
for k in range(al_imag.shape[1]):
    plt.plot(epochs_al, al_imag[:, k], label=labels_al[k])
plt.title("Assembly Layer Weights (Imag)")
plt.xlabel("Epoch")
plt.ylabel("Imag part")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize="small")
plt.tight_layout()


# ==========================================================
# 3) Losses
# ==========================================================
imag_losses = np.array(imaginary_out_losses)
data_losses = np.array(data_losses)

# --- imaginary_out_losses ---
plt.figure(figsize=(6, 4))
plt.plot(imag_losses)
plt.title("Imaginary Output Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
# plt.yscale("log")
plt.tight_layout()

# --- data_losses (log scale) ---
plt.figure(figsize=(6, 4))
plt.plot(data_losses)
plt.title("Data Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.yscale("log")    # <<<<<< LOG SCALE HERE
plt.grid(True, which="both")
plt.tight_layout()

plt.show()
